In [0]:
## read from silver table 
df_crm_prod = spark.table("workspace.silver.crm_product_info_cleaned")
df_erp_prod = spark.table("workspace.silver.erp_product_details_cleaned")

In [0]:
df_crm_prod.display()

In [0]:
df_erp_prod.display()

In [0]:
from pyspark.sql.functions import regexp_replace, col

# 1. Create the standardized join key in the CRM data
df_crm_prep = df_crm_prod.withColumn(
    "join_code", 
    regexp_replace(col("product_category_code"), "-", "_")
)

# 2. Join by explicitly referencing the ERP dataframe to avoid ambiguity
df_joined = df_crm_prep.join(
    df_erp_prod, 
    df_crm_prep["join_code"] == df_erp_prod["product_category_code"], 
    how='left'
)

# 3. Clean up: Drop the helper column and the redundant ERP code column
df_joined = df_joined.drop("join_code", df_erp_prod["product_category_code"])

df_joined.display()

In [0]:
## write to gold layer 
df_joined.write.mode("overwrite").saveAsTable("workspace.gold.dim_product_info")